In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))
from paths import DATA, FPL_REPO

BASE = FPL_REPO / "data"
OUT  = DATA

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import os

In [3]:
df1 = pd.read_csv(BASE / '2025-26' / 'gws' / 'merged_gw.csv'); df1['season'] = 2026
df2 = pd.read_csv(BASE / '2024-25' / 'gws' / 'merged_gw.csv'); df2['season'] = 2025
df3 = pd.read_csv(BASE / '2023-24' / 'gws' / 'merged_gw.csv'); df3['season'] = 2024
df4 = pd.read_csv(BASE / '2022-23' / 'gws' / 'merged_gw.csv'); df4['season'] = 2023
df5 = pd.read_csv(BASE / '2021-22' / 'gws' / 'merged_gw.csv'); df5['season'] = 2022
df6 = pd.read_csv(BASE / '2020-21' / 'gws' / 'merged_gw.csv'); df6['season'] = 2021
df = pd.concat([df1, df2, df3, df4, df5, df6], ignore_index=True)
print(f"Total rows: {len(df)}")

Total rows: 156000


In [4]:
df_row = df[['name', 'position', 'assists', 'bonus', 'clean_sheets', 'creativity',
             'goals_conceded', 'goals_scored', 'own_goals', 'penalties_missed',
             'penalties_saved', 'red_cards', 'value', 'yellow_cards',
             'defensive_contribution', 'GW', 'season', 'total_points',
             'starts', 'opponent_team', 'minutes', 'team', 'was_home']].copy()

In [5]:
df_row['player_id'] = df_row['name'].astype('category').cat.codes
player_lookup = df_row[['player_id', 'name']].drop_duplicates().reset_index(drop=True)
os.makedirs(OUT, exist_ok=True)
player_lookup.to_csv(rf'{OUT}\player_lookup.csv', index=False)
print(f"Unique players: {df_row['player_id'].nunique()}")

Unique players: 2135


In [6]:
df_row['defensive_contribution'] = df_row.groupby('position')['defensive_contribution'].transform(
    lambda x: x.fillna(x.median()))
df_row['starts'] = df_row['starts'].fillna((df_row['minutes'] >= 60).astype(int))

In [7]:
teams_2425 = pd.read_csv(rf'{BASE}\2024-25\teams.csv')[['id','name']].rename(columns={'id':'team','name':'team_name'})
teams_2526 = pd.read_csv(rf'{BASE}\2025-26\teams.csv')[['id','name']].rename(columns={'id':'team','name':'team_name'})
teams_2425['season'] = '2024-25'
teams_2526['season'] = '2025-26'
master_teams = pd.read_csv(rf'{BASE}\master_team_list.csv')
master_teams = pd.concat([master_teams, teams_2425, teams_2526], ignore_index=True)

In [8]:
season_str_map = {2026:'2025-26', 2025:'2024-25', 2024:'2023-24',
                  2023:'2022-23', 2022:'2021-22', 2021:'2020-21'}
df_row['season_str'] = df_row['season'].map(season_str_map)
df_row['opponent_team'] = df_row['opponent_team'].astype(int)
master_teams['team'] = master_teams['team'].astype(int)
master_teams['season'] = master_teams['season'].astype(str)
master_teams = master_teams.rename(columns={'team':'team_id', 'team_name':'opponent_team_name'})
df_row = df_row.merge(master_teams, left_on=['opponent_team','season_str'],
                      right_on=['team_id','season'], how='left')
df_row = df_row.drop(columns=['team_id', 'season_str', 'opponent_team', 'season_y'], errors='ignore')
df_row = df_row.rename(columns={'season_x': 'season'}) if 'season_x' in df_row.columns else df_row
print(f"Opponent NaN: {df_row['opponent_team_name'].isna().sum()}")

Opponent NaN: 0


In [9]:
df_row['position'] = df_row['position'].replace({'GKP': 'GK', 'AM': 'MID'})
lb = LabelEncoder()
df_row['position'] = lb.fit_transform(df_row['position'])
 
all_teams = pd.concat([df_row['team'], df_row['opponent_team_name']]).unique()
lb2 = LabelEncoder()
lb2.fit(all_teams)
team_encoding = pd.DataFrame({
    'team_name': lb2.classes_,
    'encoded': range(len(lb2.classes_))
})
team_encoding.to_csv(OUT / 'team_encoding.csv', index=False)
df_row['team'] = lb2.transform(df_row['team'])
df_row['opponent_team_name'] = lb2.transform(df_row['opponent_team_name'])
 
df_row['was_home'] = df_row['was_home'].map({True: 1, False: 0})
df_row = df_row.drop(columns=['name'])

In [10]:
df_row = df_row.sort_values(['player_id', 'season', 'GW'])
df_row['avg_minutes_last3'] = df_row.groupby('player_id')['minutes'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean())
df_row['avg_starts_last3'] = df_row.groupby('player_id')['starts'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean())
df_row['avg_points_last3'] = df_row.groupby('player_id')['total_points'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean())
 
df_row['avg_minutes_last3'] = df_row['avg_minutes_last3'].fillna(0)
df_row['avg_starts_last3']  = df_row['avg_starts_last3'].fillna(0)
df_row['avg_points_last3']  = df_row['avg_points_last3'].fillna(0)

In [11]:
hist = df_row[df_row['season'] < 2026]
avg_vs_opp = hist.groupby(['player_id','opponent_team_name'])['total_points'].mean().reset_index()
avg_vs_opp.columns = ['player_id', 'opponent_team_name', 'avg_points_vs_opponent']
df_row = df_row.merge(avg_vs_opp, on=['player_id','opponent_team_name'], how='left')

In [12]:
player_avg = hist.groupby('player_id')['total_points'].mean()
df_row['player_avg_tmp'] = df_row['player_id'].map(player_avg)
df_row['avg_points_vs_opponent'] = df_row['avg_points_vs_opponent'].fillna(df_row['player_avg_tmp'])
df_row['avg_points_vs_opponent'] = df_row['avg_points_vs_opponent'].fillna(df_row['avg_points_vs_opponent'].median())
df_row = df_row.drop(columns=['player_avg_tmp'])

In [13]:
avg_gc = hist.groupby('opponent_team_name')['goals_conceded'].mean().reset_index()
avg_gc.columns = ['opponent_team_name', 'avg_goals_conceded_opponent']
df_row = df_row.merge(avg_gc, on='opponent_team_name', how='left')
 
avg_cs = hist.groupby('opponent_team_name')['clean_sheets'].mean().reset_index()
avg_cs.columns = ['opponent_team_name', 'avg_clean_sheets_opponent']
df_row = df_row.merge(avg_cs, on='opponent_team_name', how='left')
 
avg_gc_away = hist[hist['was_home']==0].groupby('opponent_team_name')['goals_conceded'].mean().reset_index()
avg_gc_away.columns = ['opponent_team_name', 'avg_goals_conceded_away']
df_row = df_row.merge(avg_gc_away, on='opponent_team_name', how='left')
 
avg_gc_home = hist[hist['was_home']==1].groupby('opponent_team_name')['goals_conceded'].mean().reset_index()
avg_gc_home.columns = ['opponent_team_name', 'avg_goals_conceded_home']
df_row = df_row.merge(avg_gc_home, on='opponent_team_name', how='left')
 
for col in ['avg_goals_conceded_opponent','avg_clean_sheets_opponent',
            'avg_goals_conceded_away','avg_goals_conceded_home']:
    df_row[col] = df_row[col].fillna(df_row[col].median())


In [14]:
df_model = df_row.copy()
df_model['defensive_contribution'] = df_model.groupby('position')['defensive_contribution'].transform(
    lambda x: x.fillna(x.median()))
print(f"\nFinal shape: {df_model.shape}")
print(f"NaN values:\n{df_model.isna().sum()}")


 
df_model.to_csv(rf'{OUT}\df_model.csv', index=False) 
print(f"\nSaved df_model.csv")


Final shape: (156000, 31)
NaN values:
position                       0
assists                        0
bonus                          0
clean_sheets                   0
creativity                     0
goals_conceded                 0
goals_scored                   0
own_goals                      0
penalties_missed               0
penalties_saved                0
red_cards                      0
value                          0
yellow_cards                   0
defensive_contribution         0
GW                             0
season                         0
total_points                   0
starts                         0
minutes                        0
team                           0
was_home                       0
player_id                      0
opponent_team_name             0
avg_minutes_last3              0
avg_starts_last3               0
avg_points_last3               0
avg_points_vs_opponent         0
avg_goals_conceded_opponent    0
avg_clean_sheets_opponent      0
avg_

In [15]:
df_row["team"]

0          6
1          6
2          6
3          6
4          6
          ..
155995    26
155996    26
155997    26
155998    26
155999    26
Name: team, Length: 156000, dtype: int64